# Databricks notebook source
# MAGIC %md
# MAGIC # 01 — Source Profiling
# MAGIC Profiles the current Bronze layer: record counts, column structures, null
# MAGIC patterns, and duplicate patterns per asset class / exchange. Run this whenever
# MAGIC bronze changes (e.g. after the Phase 1 URL fix) to re-verify assumptions before
# MAGIC building on top of it — don't assume the numbers in the kickoff doc still hold
# MAGIC once the source data itself has changed.

In [0]:
%run /Workspace/Users/shreyash270204@outlook.com/databricks/utilities/config.py

Config loaded. STORAGE_ACCOUNT=financestorage1 SNAPSHOT_DATE=latest per exchange TAXONOMY_VERSION=v1


In [0]:
from pyspark.sql import functions as F

In [0]:
def profile_asset_class(asset_class: str):
    rows = []
    for exch, classes in EXCHANGE_ASSET_COVERAGE.items():
        if asset_class not in classes:
            continue
        path = latest_snapshot_path(asset_class, exch)
        if path is None:
            rows.append({"asset_class": asset_class, "exchange": exch, "status": "NO SNAPSHOT FOUND"})
            continue

        df = (
        spark.read.option("header", True).option("inferSchema", True)
        .option("multiLine", True).option("escape", "\"")
        .csv(f"{path}/*.csv")
        )
        total = df.count()
        cols = df.columns

        null_pcts = {}
        if total > 0:
            null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).first().asDict()
            null_pcts = {c: round(100 * null_counts[c] / total, 2) for c in cols}

        dup_count = 0
        if "symbol" in cols:
            dup_count = (
                df.groupBy("symbol").count().where("count > 1")
                .agg(F.sum(F.col("count") - 1)).first()[0] or 0
            )

        rows.append({
            "asset_class": asset_class,
            "exchange": exch,
            "snapshot": path.split("snapshot_date=")[-1],
            "row_count": total,
            "column_count": len(cols),
            "columns": ", ".join(cols),
            "duplicate_symbols": int(dup_count),
            "null_pct_by_column": str(null_pcts),
            "status": "OK",
        })
    return rows

In [0]:
all_rows = []
for ac in ASSET_CLASSES:
    all_rows.extend(profile_asset_class(ac))

profile_df = spark.createDataFrame(all_rows)
display(profile_df.select("asset_class", "exchange", "snapshot", "row_count", "column_count", "duplicate_symbols", "status"))

asset_class,exchange,snapshot,row_count,column_count,duplicate_symbols,status
equities,BER,2026-08-21,7767,22,0,OK
equities,BSE,2026-08-21,3856,22,0,OK
equities,FRA,2026-08-21,11587,22,0,OK
equities,GER,2026-08-21,1189,22,0,OK
equities,JPX,2026-08-21,3061,22,0,OK
equities,LSE,2026-08-21,3391,22,0,OK
equities,NSE,2026-08-21,1930,22,0,OK
equities,SHZ,2026-08-21,2248,22,0,OK
equities,VIE,2026-08-21,1355,22,0,OK
equities,NYQ,2026-08-21,4165,22,0,OK


# MAGIC %md
# MAGIC ## Aggregate null-rate check for the two columns the kickoff doc calibrated
# MAGIC data-quality thresholds against (~6% null country, ~8.5% null sector for
# MAGIC equities). Re-verify these still hold on the corrected data — the DQ framework
# MAGIC in `06_data_quality` will use whatever this notebook confirms, not the
# MAGIC original assumed numbers, if they've drifted.

In [0]:
equities_path_check = [
    latest_snapshot_path("equities", exch)
    for exch in EXCHANGE_ASSET_COVERAGE
    if "equities" in EXCHANGE_ASSET_COVERAGE[exch]
]
equities_frames = [
    spark.read.option("header", True).option("inferSchema", True)
    .option("multiLine", True).option("escape", "\"")
    .csv(f"{p}/*.csv")
    for p in equities_path_check if p
]

equities_all = equities_frames[0]
for f in equities_frames[1:]:
    equities_all = equities_all.unionByName(f, allowMissingColumns=True)

total_eq = equities_all.count()
null_country_pct = round(100 * equities_all.where(F.col("country").isNull()).count() / total_eq, 2)
null_sector_pct = round(100 * equities_all.where(F.col("sector").isNull()).count() / total_eq, 2)

print(f"Equities total rows: {total_eq}")
print(f"Null country %: {null_country_pct}  (kickoff doc assumed ~6%)")
print(f"Null sector %:  {null_sector_pct}  (kickoff doc assumed ~8.5%)")

Equities total rows: 40549
Null country %: 5.37  (kickoff doc assumed ~6%)
Null sector %:  7.17  (kickoff doc assumed ~8.5%)
